# Ejemplo 2 - Agente intermedio

## Propósito didáctico
Este notebook muestra cómo pasar de un chatbot simple a un **agente con herramientas**, capaz de decidir cuándo buscar información o cuándo usar una calculadora.

## Competencias que se trabajan
- Comprender qué es una **tool** o herramienta.
- Distinguir entre responder “solo con el modelo” y responder “con apoyo de herramientas”.
- Construir un agente moderno con `create_agent(...)`.
- Evaluar ventajas, riesgos y límites de un agente más autónomo.

## Idea general
Un modelo conversacional básico solo genera texto.  
Un agente, en cambio, puede **elegir acciones** para resolver mejor una tarea, por ejemplo:
- buscar información;
- realizar operaciones;
- consultar una base de conocimiento;
- llamar a una API.

En este ejemplo usaremos dos herramientas:
1. una búsqueda web simple;
2. una calculadora segura.

## 1. Instalación
Ejecuta esta celda una sola vez.

In [ ]:
#
!pip install -qU langchain langchain-core langchain-groq langchain-community duckduckgo-search

!pip install -U ddgs langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00


## 2. Configurar credenciales
Como en el ejemplo básico, la clave se ingresa de forma segura.

In [ ]:
#
import os
import getpass

#os.environ["GROQ_API_KEY"] = "..."

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("🔑 Ingresa tu Groq API key: ")


🔑 Ingresa tu Groq API key: ··········


## 3. Modelo
En esta sección configuramos el modelo base que razonará sobre la tarea y decidirá cuándo usar herramientas.

### Pregunta de reflexión
¿Por qué no siempre conviene dejar que el modelo responda “de memoria”?

In [ ]:
#
from langchain_groq import ChatGroq

MODEL_NAME = "llama-3.3-70b-versatile"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=1024,
    max_retries=2,
)


## 4. Herramientas
Aquí definimos las herramientas del agente.

### ¿Qué hace cada una?
- **Búsqueda**: recupera información externa.
- **Calculadora**: resuelve operaciones con más confiabilidad que el modelo generando texto libremente.

### Punto importante
La calculadora está implementada de manera **segura**, evitando el uso directo de `eval(...)`.

In [ ]:
#

from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
import ast
import operator as op

search_tool = DuckDuckGoSearchRun()

_ALLOWED_OPERATORS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.Mod: op.mod,
    ast.USub: op.neg,
}


def safe_eval(expr: str):
    def _eval(node):
        if isinstance(node, ast.Constant):
            if isinstance(node.value, (int, float)):
                return node.value
            raise TypeError("Solo se permiten números.")
        if isinstance(node, ast.BinOp):
            if type(node.op) not in _ALLOWED_OPERATORS:
                raise TypeError("Operación no permitida.")
            return _ALLOWED_OPERATORS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            if type(node.op) not in _ALLOWED_OPERATORS:
                raise TypeError("Operación unaria no permitida.")
            return _ALLOWED_OPERATORS[type(node.op)](_eval(node.operand))
        raise TypeError("Expresión no permitida.")

    parsed = ast.parse(expr, mode="eval")
    return _eval(parsed.body)


@tool
def busqueda_web(query: str) -> str:
    """Busca información general en internet cuando se necesiten datos externos o recientes."""
    return search_tool.run(query)


@tool
def calculadora(expr: str) -> str:
    """Evalúa expresiones matemáticas seguras, por ejemplo: 23*56 o (10+2)/3."""
    try:
        return str(safe_eval(expr))
    except Exception as e:
        return f"Error en el cálculo: {e}"


## 5. Crear el agente
Este es el paso donde combinamos:
- el modelo,
- las herramientas,
- y las instrucciones generales de comportamiento.

En versiones actuales de LangChain, el patrón recomendado es `create_agent(...)`.

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """Eres un asistente útil que puede usar herramientas.
- Usa la búsqueda web sólo cuando la pregunta requiera información externa o reciente.
- Usa la calculadora para operaciones numéricas.
- Si puedes responder sin herramientas, hazlo.
- Responde en español."""

agent = create_agent(
    model=llm,
    tools=[busqueda_web, calculadora],
    system_prompt=SYSTEM_PROMPT,
)


## 6. Función auxiliar para invocar el agente
La función `run_agent(...)` simplifica las pruebas y deja el código más legible.

In [ ]:
def run_agent(user_input: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
    last_message = result["messages"][-1]
    return getattr(last_message, "content", str(last_message))


## 7. Pruebas rápidas
Usa estos ejemplos para observar la diferencia entre tipos de tareas:
- una consulta factual reciente;
- un cálculo matemático;
- una pregunta que combine ambas cosas.

### Ejercicio sugerido
Compara el comportamiento del agente con el de un chatbot básico:
- ¿cuándo responde mejor?
- ¿cuándo tarda más?
- ¿en qué situaciones parece más confiable?

In [ ]:
print("=== Agente intermedio ===")
print(run_agent("¿Cuál es la capital de Japón y cuánto es 23 * 56?"))


=== Agente intermedio ===
La capital de Japón es Tokio. El resultado de la operación 23 * 56 es 1288.


## 8. Modo interactivo opcional
Este bloque te permite conversar libremente con el agente.

In [ ]:
while True:
    pregunta = input("Tú: ").strip()
    if pregunta.lower() in ["salir", "exit"]:
        print("Agente: ¡Hasta luego!")
        break
    print(f"Agente: {run_agent(pregunta)}")


Tú: hola
Agente: Hola, ¿en qué puedo ayudarte hoy?
Tú: exit
Agente: ¡Hasta luego!


## 9. Análisis didáctico
### ¿Qué mejora respecto al agente básico?
- Puede apoyarse en herramientas.
- Reduce errores en cálculos sencillos.
- Puede acceder a información más allá del contexto inmediato.

### ¿Qué riesgos aparecen?
- Dependencia de herramientas externas.
- Resultados variables según la calidad de la búsqueda.
- Mayor complejidad del flujo.
- Posibles costos y latencia más altos.

## 10. Actividad sugerida
Agrega una tercera herramienta. Algunas opciones:
- traductor;
- conversor de unidades;
- lector de archivos;
- buscador en una base local.

## 11. Cierre
Este notebook ilustra un cambio importante: pasar de “pedirle cosas a un modelo” a “diseñar un sistema que combine modelo + herramientas”.

## 12. Ideas para mejorarlo más

## 9. Ideas para mejorarlo más
- Agregar memoria persistente.
- Incorporar herramientas de archivos o RAG.
- Añadir validación estructurada de salidas.
- Implementar streaming y trazabilidad con LangSmith.